In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Modo de teste local do Qiskit Runtime

<span id="test-locally"></span>


<details>
<summary><b>Versões dos pacotes</b></summary>

O código desta página foi desenvolvido com os seguintes requisitos.
Recomendamos usar estas versões ou mais recentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>

Use o modo de teste local (disponível no `qiskit-ibm-runtime` v0.22.0 ou posterior) para testar programas antes de ajustá-los e enviá-los para hardware quântico real. Após usar o modo de teste local para verificar seu programa, tudo que você precisa mudar é o nome do backend para executá-lo em um QPU.

Para usar o modo de teste local, especifique um dos backends falsos de ``qiskit_ibm_runtime.fake_provider`` ou especifique um backend do Qiskit Aer ao instanciar uma primitiva do Qiskit Runtime ou uma sessão.

- **Backends falsos**: Os [backends falsos](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider) em ``qiskit_ibm_runtime.fake_provider`` imitam o comportamento dos QPUs da IBM&reg; usando snapshots de QPU. Os snapshots de QPU contêm informações importantes sobre o QPU, como o mapa de acoplamento, portas de base e propriedades dos qubits, que são úteis para testar o transpilador e realizar simulações ruidosas do QPU. O modelo de ruído do snapshot é aplicado automaticamente durante a simulação.
- **Simulador Aer**: Os simuladores do [Qiskit Aer](/guides/simulate-with-qiskit-aer) oferecem simulação de maior desempenho, capaz de lidar com circuitos maiores e [modelos de ruído personalizados](/guides/build-noise-models). Uma lista de opções de métodos de simulação está disponível quando você usa o `AerSimulator` no modo de teste local. Veja o [exemplo do modo de simulação Clifford](#clifford-sim), que demonstra como simular eficientemente circuitos Clifford com um grande número de qubits.

    <details>
    <summary>**Lista de métodos de simulação disponíveis no Qiskit Aer**</summary>

    Consulte a documentação do [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) para mais informações.

    * ``"automatic"``: Método de simulação padrão. Seleciona o método de simulação automaticamente, com base no circuito e no modelo de ruído.

    * ``"statevector"``: Uma simulação de vetor de estado denso que pode amostrar resultados de medição de circuitos *ideais* com todas as medições no final do circuito. Para simulações ruidosas, cada shot amostra aleatoriamente um circuito ruidoso a partir do modelo de ruído.

    * ``"density_matrix"``: Uma simulação de matriz densidade que pode amostrar resultados de medição de circuitos *ruidosos* com todas as medições no final do circuito.

    * ``"stabilizer"``: Um simulador eficiente de estado estabilizador Clifford que pode simular circuitos Clifford ruidosos, desde que todos os erros no modelo de ruído também sejam erros Clifford.

    * ``"extended_stabilizer"``: Um simulador aproximado para circuitos Clifford + T baseado na decomposição do estado em um estado estabilizador ranqueado. O número de termos cresce com o número de portas não-Clifford (T).

    * ``"matrix_product_state"``: Um simulador de vetor de estado em rede tensorial que usa uma representação de Estado Produto Matricial (MPS) para o estado. Isso pode ser feito com ou sem truncamento das dimensões de vínculo do MPS, dependendo das opções do simulador. O comportamento padrão é sem truncamento.

    * ``"unitary"``: Uma simulação de matriz unitária densa de um circuito ideal. Isso simula a matriz unitária do próprio circuito, em vez da evolução de um estado quântico inicial. Este método só pode simular portas; não suporta medição, reset ou ruído.

    * ``"superop"``: Uma simulação de matriz superoperador densa de um circuito ideal ou ruidoso. Isso simula a matriz superoperador do próprio circuito, em vez da evolução de um estado quântico inicial. Este método pode simular portas e resets ideais e ruidosos, mas não suporta medições.

    * ``"tensor_network"``: Uma simulação baseada em rede tensorial que suporta tanto vetor de estado quanto matriz densidade. Atualmente, isso está disponível apenas para GPU e é acelerado com as APIs `cuTensorNet` do cuQuantum.
    </details>

> **Note:** - Você pode especificar todas as opções do Qiskit Runtime no modo de teste local. No entanto, todas as opções, exceto shots, são ignoradas quando executadas em um simulador local.
> - Recomenda-se instalar o Qiskit Aer antes de usar backends falsos ou simuladores Aer, executando `pip install qiskit-aer`. Os backends falsos usam simuladores Aer internamente, quando disponíveis, para aproveitar seu desempenho.



## Exemplo com backends falsos

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## Exemplos com AerSimulator {#aerSimulator-examples}

Exemplo com sessões, sem ruído:

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

Para simular com ruído, especifique um QPU (hardware quântico) e envie-o para o Aer. O Aer constrói um modelo de ruído com base nos dados de calibração desse QPU e instancia um backend Aer com esse modelo. Se preferir, você pode [construir um modelo de ruído](/guides/build-noise-models).

> **Caution:** Um QPU pode ser afetado por diferentes tipos de ruído. O modelo de ruído do Qiskit Aer usado aqui simula apenas alguns deles e, portanto, tende a ser menos severo do que o ruído em um QPU real.
> 
> Para detalhes sobre quais erros são incluídos ao inicializar um modelo de ruído a partir de um QPU, consulte a referência da API Aer [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend).

Exemplo com ruído:

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>

### Simulação Clifford
Como os circuitos Clifford podem ser simulados eficientemente com resultados verificáveis, a simulação Clifford é uma ferramenta muito útil. Para um exemplo detalhado, consulte [Simulação eficiente de circuitos estabilizadores com primitivas do Qiskit Aer](/guides/simulate-stabilizer-circuits).

Exemplo:

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## Próximos passos
> **Tip:** - Revise exemplos detalhados de [primitivas.](/guides/primitives-examples)
>     - Leia [Migrar para primitivas V2](/guides/v2-primitives).
>     - Pratique com primitivas trabalhando na [lição de função de custo](/learning/courses/variational-algorithm-design/cost-functions) no IBM Quantum Learning.
>     - Aprenda como transpilar localmente na seção [Transpilar](/guides/transpile).
>     - Experimente o tutorial [Comparar configurações do transpilador](/guides/circuit-transpilation-settings#compare-transpiler-settings).